# Pandas Cheat Sheet

Covers everything you need for a Data Engineer interview:
1. Creating DataFrames
2. Inspecting Data
3. Selecting & Filtering
4. Adding, Renaming & Dropping
5. Sorting & Ranking
6. GroupBy & Aggregation
7. Merging & Joining
8. Pivot Tables & Crosstab
9. Handling Missing Data
10. String Operations
11. DateTime Operations
12. Apply, Map & Lambda
13. Window Functions (rolling, expanding, rank)
14. Reading & Writing Files
15. Performance Tips

---
## Setup

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 100)
print(pd.__version__)

---
## 1. Creating DataFrames

In [ ]:
# From dict
df = pd.DataFrame({
    'name':   ['Alice','Bob','Charlie','Diana','Eve'],
    'dept':   ['Eng','Data','Eng','HR','Data'],
    'salary': [90000, 85000, 95000, 70000, 88000],
    'age':    [30, 25, 35, 28, 32],
    'joined': ['2020-01-15','2019-06-01','2018-03-20','2021-09-10','2020-11-05']
})
df['joined'] = pd.to_datetime(df['joined'])
print(df)

In [ ]:
# From list of tuples
data = [('Alice',90000), ('Bob',85000)]
df2 = pd.DataFrame(data, columns=['name','salary'])
print(df2)

# From range / numpy
df3 = pd.DataFrame(np.arange(12).reshape(3,4), columns=['a','b','c','d'])
print(df3)

# Empty DataFrame with schema
empty = pd.DataFrame(columns=['id','value','timestamp'])
print(empty.dtypes)

---
## 2. Inspecting Data

In [ ]:
print(df.shape)          # (rows, cols)
print(df.dtypes)         # column types
print(df.columns.tolist())
print(df.index.tolist())
df.head(3)

In [ ]:
df.tail(2)

In [ ]:
df.info()

In [ ]:
df.describe()            # stats for numeric cols

In [ ]:
df.describe(include='all')  # stats for all cols

In [ ]:
print(df.nunique())      # unique count per column
print(df['dept'].value_counts())
print(df['dept'].unique())

---
## 3. Selecting & Filtering

In [ ]:
# Select columns
print(df['name'])           # Series
print(df[['name','salary']]) # DataFrame

# loc — label based
print(df.loc[0])            # row by index label
print(df.loc[0:2, 'name':'salary'])  # row slice, col slice

# iloc — position based
print(df.iloc[0])           # first row
print(df.iloc[0:3, 0:2])    # first 3 rows, first 2 cols
print(df.iloc[-1])          # last row

In [ ]:
# Boolean filtering
print(df[df['salary'] > 87000])
print(df[df['dept'] == 'Eng'])

# Multiple conditions — use & | ~
print(df[(df['salary'] > 85000) & (df['dept'] == 'Eng')])
print(df[(df['dept'] == 'Eng') | (df['dept'] == 'Data')])
print(df[~(df['dept'] == 'HR')])   # NOT

# isin
print(df[df['dept'].isin(['Eng','Data'])])

# between
print(df[df['salary'].between(85000, 92000)])

# query — SQL-like syntax
print(df.query('salary > 87000 and dept == "Eng"'))

---
## 4. Adding, Renaming & Dropping

In [ ]:
# Add column
df['bonus'] = df['salary'] * 0.10
df['senior'] = df['age'] > 30
print(df.head())

# Rename columns
df_r = df.rename(columns={'name':'employee','dept':'department'})
print(df_r.columns.tolist())

# Drop columns
df_d = df.drop(columns=['bonus','senior'])
print(df_d.columns.tolist())

# Drop rows by index
df_dr = df.drop(index=[0,1])
print(df_dr)

# Reset index
df_reset = df_dr.reset_index(drop=True)
print(df_reset)

---
## 5. Sorting & Ranking

In [ ]:
# Sort by single column
print(df.sort_values('salary', ascending=False))

# Sort by multiple columns
print(df.sort_values(['dept','salary'], ascending=[True, False]))

# Rank
df['salary_rank'] = df['salary'].rank(ascending=False, method='dense').astype(int)
print(df[['name','salary','salary_rank']].sort_values('salary_rank'))

# nlargest / nsmallest
print(df.nlargest(3, 'salary'))
print(df.nsmallest(2, 'salary'))

---
## 6. GroupBy & Aggregation

In [ ]:
# Basic groupby
print(df.groupby('dept')['salary'].mean())
print(df.groupby('dept')['salary'].agg(['mean','min','max','count']))

# Multiple columns
print(df.groupby('dept').agg(
    avg_salary=('salary','mean'),
    total_salary=('salary','sum'),
    headcount=('name','count'),
    avg_age=('age','mean')
).round(2))

In [ ]:
# groupby with filter — depts with avg salary > 85000
dept_avg = df.groupby('dept')['salary'].mean()
print(dept_avg[dept_avg > 85000])

# transform — broadcast group stat back to original df
df['dept_avg_salary'] = df.groupby('dept')['salary'].transform('mean')
print(df[['name','dept','salary','dept_avg_salary']])

# Salary vs dept average
df['vs_avg'] = df['salary'] - df['dept_avg_salary']
print(df[['name','dept','salary','vs_avg']])

In [ ]:
# cumulative sum within group (like SQL SUM OVER PARTITION BY ORDER BY)
df_sorted = df.sort_values(['dept','salary'])
df_sorted['cumulative_salary'] = df_sorted.groupby('dept')['salary'].cumsum()
print(df_sorted[['name','dept','salary','cumulative_salary']])

---
## 7. Merging & Joining

> Equivalent to SQL JOINs

In [ ]:
dept_info = pd.DataFrame({
    'dept':     ['Eng','Data','HR','Finance'],
    'location': ['Seattle','NYC','Chicago','Boston'],
    'budget':   [500000, 300000, 150000, 400000]
})

# INNER JOIN (default)
inner = pd.merge(df, dept_info, on='dept', how='inner')
print(inner[['name','dept','salary','location']])

# LEFT JOIN
left = pd.merge(df, dept_info, on='dept', how='left')
print(left[['name','dept','location']].head())

# RIGHT JOIN
right = pd.merge(df, dept_info, on='dept', how='right')
print(right[['name','dept','location']])

# OUTER JOIN
outer = pd.merge(df, dept_info, on='dept', how='outer')
print(outer[['name','dept','location']])

In [ ]:
# Join on different column names
df_left  = pd.DataFrame({'emp_id':[1,2,3],'name':['A','B','C']})
df_right = pd.DataFrame({'id':[1,2,4],'score':[90,85,88]})
print(pd.merge(df_left, df_right, left_on='emp_id', right_on='id', how='left'))

# concat — stack DataFrames vertically or horizontally
df_a = pd.DataFrame({'x':[1,2],'y':[3,4]})
df_b = pd.DataFrame({'x':[5,6],'y':[7,8]})
print(pd.concat([df_a, df_b], ignore_index=True))          # vertical
print(pd.concat([df_a, df_b], axis=1))                     # horizontal

---
## 8. Pivot Tables & Crosstab

In [ ]:
# pivot_table — like SQL GROUP BY with multiple dimensions
sales = pd.DataFrame({
    'region':  ['North','South','North','South','North'],
    'product': ['A','A','B','B','A'],
    'month':   ['Jan','Jan','Jan','Jan','Feb'],
    'revenue': [100,150,200,130,120]
})

pivot = pd.pivot_table(sales, values='revenue',
                       index='region', columns='product',
                       aggfunc='sum', fill_value=0)
print(pivot)

# Add margins (totals)
pivot_total = pd.pivot_table(sales, values='revenue',
                              index='region', columns='product',
                              aggfunc='sum', fill_value=0, margins=True)
print(pivot_total)

In [ ]:
# crosstab — frequency table
print(pd.crosstab(df['dept'], df['senior']))

# melt — wide to long (unpivot)
wide = pd.DataFrame({'name':['Alice','Bob'],'q1':[100,90],'q2':[110,95]})
long = pd.melt(wide, id_vars='name', var_name='quarter', value_name='sales')
print(long)

---
## 9. Handling Missing Data

In [ ]:
df_null = pd.DataFrame({
    'a': [1, None, 3, None, 5],
    'b': [10, 20, None, 40, 50],
    'c': ['x', 'y', None, 'w', 'v']
})

print(df_null.isnull().sum())          # count nulls per column
print(df_null.isnull().sum().sum())    # total nulls
print(df_null.notnull().sum())

# Drop rows/cols with nulls
print(df_null.dropna())                # drop rows with any null
print(df_null.dropna(how='all'))       # drop rows where ALL are null
print(df_null.dropna(subset=['a']))    # drop rows where 'a' is null
print(df_null.dropna(axis=1))         # drop columns with any null

In [ ]:
# Fill nulls
print(df_null.fillna(0))                          # fill all with 0
print(df_null.fillna({'a':0,'b':df_null['b'].mean(),'c':'unknown'}))

# Forward fill / backward fill
print(df_null.ffill())   # propagate last valid value forward
print(df_null.bfill())   # propagate next valid value backward

# Interpolate numeric
df_interp = pd.DataFrame({'val':[1, None, None, 4, 5]})
print(df_interp.interpolate())

---
## 10. String Operations (str accessor)

In [ ]:
s = pd.Series(['  Alice Smith ', 'bob jones', 'CHARLIE BROWN', 'diana_prince'])

print(s.str.strip())
print(s.str.lower())
print(s.str.upper())
print(s.str.title())
print(s.str.replace('_',' '))
print(s.str.contains('alice', case=False, na=False))
print(s.str.startswith(' '))
print(s.str.len())
print(s.str.split().str[0])          # first word
print(s.str.extract(r'([A-Za-z]+)')) # regex extract first word

In [ ]:
# String filtering on DataFrame
print(df[df['name'].str.startswith('A')])
print(df[df['name'].str.contains('li', case=False)])
print(df['name'].str.lower().str.replace(' ','_'))

---
## 11. DateTime Operations

In [ ]:
df['joined'] = pd.to_datetime(df['joined'])

# Extract components
df['year']  = df['joined'].dt.year
df['month'] = df['joined'].dt.month
df['day']   = df['joined'].dt.day
df['dow']   = df['joined'].dt.day_name()   # day of week
print(df[['name','joined','year','month','dow']])

# Tenure in days
df['tenure_days'] = (pd.Timestamp.today() - df['joined']).dt.days
print(df[['name','joined','tenure_days']])

# Date arithmetic
df['review_date'] = df['joined'] + pd.DateOffset(years=1)
print(df[['name','joined','review_date']].head())

In [ ]:
# Date range and resampling
dates = pd.date_range('2024-01-01', periods=12, freq='ME')
ts = pd.Series(np.random.randint(100,200,12), index=dates, name='revenue')
print(ts)

# Resample to quarterly
print(ts.resample('QE').sum())
print(ts.resample('QE').mean().round(2))

---
## 12. Apply, Map & Lambda

In [ ]:
# apply on Series
df['salary_band'] = df['salary'].apply(lambda x: 'High' if x >= 90000 else 'Mid' if x >= 80000 else 'Low')
print(df[['name','salary','salary_band']])

# apply on DataFrame rows (axis=1)
df['summary'] = df.apply(lambda row: f"{row['name']} ({row['dept']})", axis=1)
print(df['summary'])

# map — element-wise on Series (for simple value replacement)
dept_code = {'Eng':'E','Data':'D','HR':'H'}
df['dept_code'] = df['dept'].map(dept_code)
print(df[['name','dept','dept_code']])

In [ ]:
# applymap / map on whole DataFrame (element-wise)
df_num = pd.DataFrame({'a':[1,2,3],'b':[4,5,6]})
print(df_num.map(lambda x: x**2))   # pandas >= 2.1 uses .map()

# np.where — vectorized if-else (faster than apply)
df['level'] = np.where(df['salary'] >= 90000, 'Senior', 'Junior')
print(df[['name','salary','level']])

# np.select — multiple conditions
conditions = [df['salary'] >= 90000, df['salary'] >= 85000]
choices    = ['L5', 'L4']
df['grade'] = np.select(conditions, choices, default='L3')
print(df[['name','salary','grade']])

---
## 13. Window Functions

> Equivalent to SQL window functions (OVER PARTITION BY)

In [ ]:
# Rolling — moving average
ts_df = pd.DataFrame({'day': range(1,8), 'sales':[10,20,15,30,25,35,40]})
ts_df['rolling_3'] = ts_df['sales'].rolling(window=3).mean()
ts_df['expanding'] = ts_df['sales'].expanding().mean()
print(ts_df)

In [ ]:
# Rank within group — equivalent to ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC)
df['rank_in_dept'] = df.groupby('dept')['salary'].rank(method='dense', ascending=False).astype(int)
print(df[['name','dept','salary','rank_in_dept']].sort_values(['dept','rank_in_dept']))

# Top 1 per group (like SQL ROW_NUMBER = 1)
top1 = df[df['rank_in_dept'] == 1][['dept','name','salary']]
print(top1)

In [ ]:
# shift — lag/lead values (like SQL LAG/LEAD)
df_s = pd.DataFrame({'month':['Jan','Feb','Mar','Apr'],'revenue':[100,120,110,140]})
df_s['prev_month'] = df_s['revenue'].shift(1)          # LAG(1)
df_s['next_month'] = df_s['revenue'].shift(-1)         # LEAD(1)
df_s['mom_change'] = df_s['revenue'] - df_s['prev_month']  # month-over-month
print(df_s)

In [ ]:
# pct_change — percentage change
df_s['pct_change'] = df_s['revenue'].pct_change().mul(100).round(2)
print(df_s)

---
## 14. Reading & Writing Files

In [ ]:
import io

# ── CSV ──
csv_data = '''name,salary,dept
Alice,90000,Eng
Bob,85000,Data
Charlie,95000,Eng'''

df_csv = pd.read_csv(io.StringIO(csv_data))
print(df_csv)

# Common read_csv options
# pd.read_csv('file.csv', sep=',', header=0, index_col=0,
#             usecols=['name','salary'], dtype={'salary':int},
#             parse_dates=['joined'], na_values=['NA','NULL'],
#             nrows=1000, skiprows=2, encoding='utf-8')

# Write CSV
buf = io.StringIO()
df_csv.to_csv(buf, index=False)
print(buf.getvalue())

In [ ]:
# ── JSON ──
import json

json_str = df_csv.to_json(orient='records', indent=2)
print(json_str)

df_json = pd.read_json(io.StringIO(json_str))
print(df_json)

# orient options: 'records', 'split', 'index', 'columns', 'values'

In [ ]:
# ── Excel (requires openpyxl) ──
# df.to_excel('output.xlsx', sheet_name='Sheet1', index=False)
# df = pd.read_excel('file.xlsx', sheet_name='Sheet1')

# ── Parquet (requires pyarrow or fastparquet) ──
# df.to_parquet('output.parquet', index=False)
# df = pd.read_parquet('file.parquet')

# ── SQL ──
# import sqlalchemy
# engine = sqlalchemy.create_engine('sqlite:///mydb.db')
# df.to_sql('table_name', engine, if_exists='replace', index=False)
# df = pd.read_sql('SELECT * FROM table_name', engine)
print('File I/O patterns shown above (commented out to avoid file creation)')

---
## 15. Performance Tips

### Use vectorized operations — avoid loops
```python
# SLOW
for i in range(len(df)):
    df.loc[i,'col'] = df.loc[i,'a'] + df.loc[i,'b']

# FAST
df['col'] = df['a'] + df['b']
```

### Prefer np.where / np.select over apply for conditionals
```python
# SLOW
df['flag'] = df['val'].apply(lambda x: 1 if x > 0 else 0)

# FAST
df['flag'] = np.where(df['val'] > 0, 1, 0)
```

### Use categorical dtype for low-cardinality string columns
```python
df['dept'] = df['dept'].astype('category')  # saves memory
```

### Downcast numeric types
```python
df['age'] = pd.to_numeric(df['age'], downcast='integer')
```

### Read only needed columns
```python
df = pd.read_csv('big.csv', usecols=['id','value','date'])
```

### Check memory usage
```python
df.memory_usage(deep=True).sum() / 1024**2  # MB
```

### Key methods summary
| Task | Method |
|------|--------|
| Filter rows | `df[condition]`, `df.query()` |
| Select cols | `df[cols]`, `df.loc`, `df.iloc` |
| Group & agg | `df.groupby().agg()` |
| Broadcast group stat | `df.groupby().transform()` |
| Join tables | `pd.merge()` |
| Stack/unstack | `pd.concat()`, `pd.melt()`, `pivot_table()` |
| Handle nulls | `fillna()`, `dropna()`, `ffill()`, `bfill()` |
| Window funcs | `rolling()`, `expanding()`, `shift()`, `rank()` |
| String ops | `str.lower/upper/contains/extract/split` |
| Date ops | `dt.year/month/day/day_name/days` |